In [25]:
import pandas as pd
import numpy as np

#Loading the dataset
df = pd.read_csv('bangalore_tech_salaries.csv')

#Info on the dataset
print("Dataframe info")
print(df.info())

#Missing values 
print("MISSING VALUES")
print(df.isnull().sum())

#Duplicates
print("Duplicates")
print(df.duplicated().sum())




Dataframe info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1015 entries, 0 to 1014
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Employee_ID     1015 non-null   object
 1   Role            1015 non-null   object
 2   years_exp       1015 non-null   int64 
 3   Current_CTC     1015 non-null   object
 4   Previous_CTC    815 non-null    object
 5   Company         1015 non-null   object
 6   company_TYPE    1015 non-null   object
 7   Skills          988 non-null    object
 8   Location        995 non-null    object
 9   Education_Tier  1015 non-null   object
 10  Joining_Year    1015 non-null   int64 
 11  Work_Mode       1015 non-null   object
dtypes: int64(2), object(10)
memory usage: 95.3+ KB
None
MISSING VALUES
Employee_ID         0
Role                0
years_exp           0
Current_CTC         0
Previous_CTC      200
Company             0
company_TYPE        0
Skills             27
Location      

In [26]:
df.columns = df.columns.str.strip().str.lower()
role_mapping = {
    'sde backend': 'SDE Backend', 'backend engineer': 'SDE Backend', 'backend dev': 'SDE Backend', 'be': 'SDE Backend', 'backend developer': 'SDE Backend',
    'data scientist': 'Data Scientist', 'ds': 'Data Scientist', 'data science': 'Data Scientist',
    'sde full-stack': 'SDE Full-Stack', 'full stack engineer': 'SDE Full-Stack', 'fs': 'SDE Full-Stack',
    'devops engineer': 'DevOps Engineer', 'devops': 'DevOps Engineer',
    'product manager': 'Product Manager', 'pm': 'Product Manager',
    'business analyst': 'Business Analyst', 'ba': 'Business Analyst',
    'ui/ux designer': 'UI/UX Designer', 'ui/ux': 'UI/UX Designer',
    'data analyst': 'Data Analyst', 'da': 'Data Analyst'
}

df['role_clean'] = df['role'].astype(str).str.strip().str.lower().map(role_mapping)

#Custom Non-Regex Parsing Function for Messy CTC Strings

def parse_ctc_to_lpa(ctc_val):
    if pd.isna(ctc_val):
        return np.nan
    
    # Clean structural strings, remove commas and currency labels
    s = str(ctc_val).upper().replace('LPA', '').replace(',', '').strip()
    
    try:
        numeric_value = float(s)
        # Trap handling: If value is in raw rupees (e.g., 1550000), scale it down to LPA format (15.5)
        if numeric_value > 100000:
            return numeric_value / 100000
        return numeric_value
    except ValueError:
        return np.nan

df['current_ctc_lpa'] = df['current_ctc'].apply(parse_ctc_to_lpa)
df['previous_ctc_lpa'] = df['previous_ctc'].apply(parse_ctc_to_lpa)

#Standardize Company Type strings
def clean_company_type(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if 'unicorn' in s: return 'Unicorn'
    if 'mnc' in s: return 'MNC'
    if 'mid' in s: return 'Mid-size'
    if 'early' in s: return 'Early-stage'
    return val

df['company_type_clean'] = df['company_type'].apply(clean_company_type)

#Standardize Education Tier strings
def clean_education_tier(val):
    if pd.isna(val):
        return np.nan
    s = str(val).strip().lower()
    if '1' in s or 't1' in s: return 'Tier 1'
    if '2' in s or 't2' in s: return 'Tier 2'
    if '3' in s or 't3' in s: return 'Tier 3'
    return val

df['education_tier_clean'] = df['education_tier'].apply(clean_education_tier)

#Remove duplicate data observations
df = df.drop_duplicates()

#Print data quality checks to verify execution metrics
print("=== CLEANED DATAFRAME DATA TYPES ===")
print(df.dtypes)
print("\n=== STANDARDIZED ROLE COUNTS ===")
print(df['role_clean'].value_counts())

=== CLEANED DATAFRAME DATA TYPES ===
employee_id              object
role                     object
years_exp                 int64
current_ctc              object
previous_ctc             object
company                  object
company_type             object
skills                   object
location                 object
education_tier           object
joining_year              int64
work_mode                object
role_clean               object
current_ctc_lpa         float64
previous_ctc_lpa        float64
company_type_clean       object
education_tier_clean     object
dtype: object

=== STANDARDIZED ROLE COUNTS ===
role_clean
SDE Backend         93
Business Analyst    76
Data Scientist      64
Product Manager     55
Data Analyst        50
DevOps Engineer     46
SDE Full-Stack      38
UI/UX Designer      23
Name: count, dtype: int64


In [27]:
#Q3.1 

role_stats = df.groupby('role_clean')['current_ctc_lpa'].agg(['median','mean','min','max'])
role_distribution = role_stats.sort_values(by='median', ascending=False)
print(role_distribution)

                     median       mean   min   max
role_clean                                        
Product Manager   32.100000  36.375510  13.8  80.1
UI/UX Designer    22.000000  21.904762   7.0  44.1
Data Scientist    21.800000  26.105660  10.9  73.5
SDE Full-Stack    21.700000  25.900000  10.7  71.7
SDE Backend       20.900000  22.601266   7.9  55.1
DevOps Engineer   19.399995  22.535714   9.7  60.3
Business Analyst  19.300000  22.011765   6.8  52.7
Data Analyst      16.300000  17.318750   5.6  43.4


In [28]:
#Q3.2

sde_backend_df = df[df['role_clean'] == 'SDE Backend'].copy()

exp_bins = [-1,1,3,5,100]
exp_labels = ['0 to 1 years','2 to 3 years','4 to 5 years','6+ years']
sde_backend_df['exp_band'] = pd.cut(sde_backend_df['years_exp'],bins=exp_bins,labels=exp_labels)

#Median metrics

backend_exp_curve = sde_backend_df.groupby('exp_band',observed=False)['current_ctc_lpa'].median().reset_index()
print(backend_exp_curve)

       exp_band  current_ctc_lpa
0  0 to 1 years             11.7
1  2 to 3 years             19.6
2  4 to 5 years             25.8
3      6+ years             40.2


In [29]:
#Q3.3
sde_roles = ['SDE Backend', 'SDE Frontend', 'SDE Full-Stack']
sde_group_df = df[df['role_clean'].isin(sde_roles)].copy()

selected_skills = ['AWS', 'ML', 'System Design', 'Kubernetes']
skill_premium_records = []

for technical_skill in selected_skills:
    # Skill presence data segment
    has_skill = sde_group_df['skills'].str.contains(technical_skill, na=False, case=False)
    
    median_with = sde_group_df[has_skill]['current_ctc_lpa'].median()
    median_without = sde_group_df[~has_skill]['current_ctc_lpa'].median()
    percentage_premium = ((median_with - median_without) / median_without) * 100
    
    skill_premium_records.append({
        'Skill': technical_skill,
        'With Skill': median_with,
        'Without': median_without,
        'Premium': percentage_premium
    })

skill_premium_df = pd.DataFrame(skill_premium_records)
print(skill_premium_df.to_string(index=False))

        Skill  With Skill  Without    Premium
          AWS        20.9    21.40  -2.336449
           ML        19.9    21.30  -6.572770
System Design        20.6    21.60  -4.629630
   Kubernetes        17.3    21.35 -18.969555


In [30]:
#Q3.4

company_type_medians = sde_backend_df.groupby('company_type_clean')['current_ctc_lpa'].median()
target_company_order = ['Unicorn', 'MNC', 'Mid-size', 'Early-stage']
company_type_medians = company_type_medians.reindex(target_company_order)

unicorn_ceiling = company_type_medians['Unicorn']
mnc_discount = ((company_type_medians['MNC'] - unicorn_ceiling) / unicorn_ceiling) * 100

print(company_type_medians)
print(f"\nMNC Gap vs Unicorn Baseline Ceiling: {mnc_discount:.2f}%")

company_type_clean
Unicorn        26.40
MNC            20.30
Mid-size       18.65
Early-stage    15.60
Name: current_ctc_lpa, dtype: float64

MNC Gap vs Unicorn Baseline Ceiling: -23.11%


In [31]:
# Synchronize experience band designations to the main dataframe
df['exp_band'] = pd.cut(df['years_exp'], bins=exp_bins, labels=exp_labels)

cohort_keys = ['role_clean', 'company_type_clean', 'exp_band']

# Calculate group counts
df['cohort_member_count'] = df.groupby(cohort_keys, observed=False)['current_ctc_lpa'].transform('count')
statistically_sound_df = df[df['cohort_member_count'] >= 10].copy()

# Calculate cohort baseline values and variances
statistically_sound_df['group_median'] = statistically_sound_df.groupby(cohort_keys, observed=False)['current_ctc_lpa'].transform('median')
statistically_sound_df['gap'] = statistically_sound_df['current_ctc_lpa'] - statistically_sound_df['group_median']

#Top 5 most underpaid professionals
top_underpaid_df = statistically_sound_df.sort_values(by='gap', ascending=True).head(5)
print(top_underpaid_df[['employee_id', 'role_clean', 'company_type_clean', 'years_exp', 'current_ctc_lpa', 'group_median', 'gap']])

    employee_id   role_clean company_type_clean  years_exp  current_ctc_lpa  \
938     BLR0525  SDE Backend                MNC          2             14.8   
94      BLR0328  SDE Backend                MNC          2             16.4   
527     BLR0157  SDE Backend                MNC          2             18.0   
184     BLR0918  SDE Backend                MNC          2             18.0   
876     BLR0965  SDE Backend                MNC          2             18.9   

     group_median   gap  
938         19.95 -5.15  
94          19.95 -3.55  
527         19.95 -1.95  
184         19.95 -1.95  
876         19.95 -1.05  


In [32]:
#FINAL SUMMARY REPORT

print("==================================================================")
print("                  BANGALORE TECH SALARY DECODER                  ")
print("==================================================================")
print("Dataset : 1,000 Bengaluru tech professionals (synthetic)")
print("Period  : 2024 employment snapshot\n")

print("MEDIAN CTC BY ROLE (in LPA)")
print("-" * 50)
for job_title, rows in role_distribution.iterrows():
    print(f"{job_title:<30} : {rows['median']:>5.1f} LPA")

print("\nSDE BACKEND CTC BY EXPERIENCE BAND")
print("-" * 50)
for index_val, rows in backend_exp_curve.iterrows():
    band_string = rows['exp_band']
    median_lpa = rows['current_ctc_lpa']
    if index_val == 0:
        print(f"{band_string:<20} : {median_lpa:>5.1f} LPA median")
    else:
        prior_median = backend_exp_curve.iloc[index_val - 1]['current_ctc_lpa']
        band_growth = ((median_lpa - prior_median) / prior_median) * 100
        print(f"{band_string:<20} : {median_lpa:>5.1f} LPA median (+{band_growth:>2.0f}% growth)")

print("\nSKILL PREMIUM FOR SDES (median CTC)")
print("-" * 50)
print(f"{'Skill':<20} {'With Skill':<15} {'Without':<12} {'Premium'}")
for _, rows in skill_premium_df.iterrows():
    print(f"{rows['Skill']:<20} {rows['With Skill']:>4.1f} LPA       {rows['Without']:>4.1f} LPA    +{rows['Premium']:+3.0f}%")

print("\nCOMPANY-TYPE PREMIUM (SDE Backend, same role)")
print("-" * 50)
print(f"Unicorn              : {company_type_medians['Unicorn']:>5.1f} LPA <- baseline ceiling")
for firm_type in ['MNC', 'Mid-size', 'Early-stage']:
    firm_value = company_type_medians[firm_type]
    firm_premium = ((firm_value - unicorn_ceiling) / unicorn_ceiling) * 100
    print(f"{firm_type:<20} : {firm_value:>5.1f} LPA ({firm_premium:>4.0f}% vs Unicorn)")

print("\nTOP 5 MOST-UNDERPAID PROFESSIONALS")
print("-" * 50)
for _, rows in top_underpaid_df.iterrows():
    print(f"{rows['employee_id']:<10} {rows['role_clean']:<16} {rows['company_type_clean']:<12} {int(rows['years_exp']):>2} yrs  gap: {rows['gap']:>5.1f} LPA")
print("==================================================================")

                  BANGALORE TECH SALARY DECODER                  
Dataset : 1,000 Bengaluru tech professionals (synthetic)
Period  : 2024 employment snapshot

MEDIAN CTC BY ROLE (in LPA)
--------------------------------------------------
Product Manager                :  32.1 LPA
UI/UX Designer                 :  22.0 LPA
Data Scientist                 :  21.8 LPA
SDE Full-Stack                 :  21.7 LPA
SDE Backend                    :  20.9 LPA
DevOps Engineer                :  19.4 LPA
Business Analyst               :  19.3 LPA
Data Analyst                   :  16.3 LPA

SDE BACKEND CTC BY EXPERIENCE BAND
--------------------------------------------------
0 to 1 years         :  11.7 LPA median
2 to 3 years         :  19.6 LPA median (+68% growth)
4 to 5 years         :  25.8 LPA median (+32% growth)
6+ years             :  40.2 LPA median (+56% growth)

SKILL PREMIUM FOR SDES (median CTC)
--------------------------------------------------
Skill                With Skill      With

Target Scaled Innovators Over Corporate Hubs:

1.According to the clean dataset metrics, Unicorn employers establish the benchmark salary ceiling at 26.4 LPA for SDE Backend roles, while traditional MNC structures subject identical positions to a sharp 23% pay discount. Action: Job-seeking students looking to maximize their entry-level CTC footprint should actively target and prioritize applications toward venture-backed scale-ups over legacy enterprise organizations.  

Capitalize on Early-Career Velocity:

2.Skill Lists Do Not Guarantee Premiums: Surprisingly, filtering the dataset shows that listing specific DevOps or System Design skills does not result in a higher median CTC for SDEs; in fact, engineers listing Kubernetes see a -19% median wage penalty (17.3 LPA vs 21.4 LPA). Action: Students should realize that core experience and company type dictate baseline pay far more heavily than simply keyword-stuffing a resume with specific toolchains.  

Guard Against the Enterprise Loyalty Discount:

3.Granular peer-group parsing surfaces extreme salary stagnation anomalies within traditional networks, where the most severely underpaid professional (BLR0525) operates at a significant -5.2 LPA deficit below their direct cohort median at just the 2-year experience mark. Action: Engineering professionals embedded inside traditional corporate tracking systems must run independent, numbers-backed market audits by their second year to negotiate structural adjustments or prepare for an external transition to correct their salary positioning.  